# Program 4: Autoencoder for Image Compression

**Course:** Deep Learning Lab — 7th Semester, CSE (AI & ML)  
**Environment:** Google Colab (Runtime → Change runtime type → GPU recommended for CNN / transfer learning)

This notebook follows Experiment 4 in the uploaded *DL Lab Manual*. Run cells from top to bottom. The program saves generated artifacts in `/content/DL_Lab_Outputs/Program_4`.

**Note:** Training accuracy, test accuracy, loss values and predictions can vary between runs and hardware. The notebook generates the actual output files when executed in Colab.

In [ ]:
# Import the package or class needed in this experiment.
import numpy as np
# Import the package or class needed in this experiment.
import pandas as pd
# Import the package or class needed in this experiment.
import tensorflow as tf
# Import the package or class needed in this experiment.
import matplotlib.pyplot as plt
# Import the package or class needed in this experiment.
from pathlib import Path
# Import the package or class needed in this experiment.
from tensorflow.keras import layers, Model

# Perform this processing step in the experiment.
tf.keras.utils.set_random_seed(42)
# Create or define the folder used to store program outputs.
OUT = Path("/content/DL_Lab_Outputs/Program_4")
# Create or define the folder used to store program outputs.
OUT.mkdir(parents=True, exist_ok=True)

# Assign the resulting value to `(x_train, _), (x_test, _)`.
(x_train, _), (x_test, _) = tf.keras.datasets.mnist.load_data()
# Assign the resulting value to `x_train`.
x_train = x_train.astype("float32") / 255.0
# Assign the resulting value to `x_test`.
x_test = x_test.astype("float32") / 255.0
# Assign the resulting value to `x_train`.
x_train = x_train.reshape(-1, 784)
# Assign the resulting value to `x_test`.
x_test = x_test.reshape(-1, 784)

# Assign the resulting value to `input_img`.
input_img = layers.Input(shape=(784,))
# Assign the resulting value to `encoded`.
encoded = layers.Dense(32, activation="relu", name="encoded")(input_img)
# Assign the resulting value to `decoded`.
decoded = layers.Dense(784, activation="sigmoid")(encoded)
# Assign the resulting value to `autoencoder`.
autoencoder = Model(input_img, decoded)
# Assign the resulting value to `encoder`.
encoder = Model(input_img, encoded)

# Configure the optimizer, loss function, and evaluation metrics.
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
# Train the model using the prepared training data.
history = autoencoder.fit(x_train, x_train, epochs=5, batch_size=256,
                          # Assign the resulting value to `shuffle`.
                          shuffle=True, validation_data=(x_test, x_test), verbose=1)

# Generate predictions for the supplied input samples.
encoded_imgs = encoder.predict(x_test[:1000], verbose=0)
# Generate predictions for the supplied input samples.
decoded_imgs = autoencoder.predict(x_test[:10], verbose=0)
# Display the current result or evaluation metric.
print("Original test image shape:", x_test.shape)
# Display the current result or evaluation metric.
print("Compressed first 1000 images:", encoded_imgs.shape)
# Display the current result or evaluation metric.
print("Compression per image: 784 values -> 32 values")

# Assign the resulting value to `n`.
n = 5
# Create, format, save, or display the visualization.
plt.figure(figsize=(10,4))
# Control execution using this condition, loop, or context.
for i in range(n):
    # Assign the resulting value to `ax`.
    ax = plt.subplot(2,n,i+1); plt.imshow(x_test[i].reshape(28,28), cmap="gray")
    # Create, format, save, or display the visualization.
    plt.title("Original"); plt.axis("off")
    # Assign the resulting value to `ax`.
    ax = plt.subplot(2,n,i+1+n); plt.imshow(decoded_imgs[i].reshape(28,28), cmap="gray")
    # Create, format, save, or display the visualization.
    plt.title("Reconstructed"); plt.axis("off")
# Create, format, save, or display the visualization.
plt.tight_layout(); plt.savefig(OUT/"original_vs_reconstructed.png", dpi=150); plt.show()
# Save experiment results to an output file.
pd.DataFrame(history.history).to_csv(OUT/"training_history.csv", index=False)
# Save experiment results to an output file.
np.save(OUT/"compressed_test_vectors.npy", encoded_imgs)
# Save the trained model or reusable objects.
autoencoder.save(OUT/"autoencoder.keras")
# Save the trained model or reusable objects.
encoder.save(OUT/"encoder.keras")